# Notebook 3 — Privacy with Anonymizer

**Hands-on synthetic data pipeline workshop with NeMo Data Designer and NeMo Anonymizer — PyData London 2026**

## What you'll build

Take a small slice of **production usage data** and turn it into something safer to inspect, share, or reuse: detect names, companies, emails, phone numbers, account IDs, addresses, and other operational identifiers in user prompts, assistant responses, and trace excerpts, then apply replacement strategies that preserve utility while reducing leakage risk.

## What you'll learn

1. **PII detection as a pipeline stage** — not an afterthought. It belongs anywhere production text might leave the system boundary.
2. **Four replacement strategies** — `Substitute`, `Redact`, `Annotate`, `Hash` — and the privacy/utility tradeoff between them.
3. **`preview()` before committing** — the Rich highlight view shows you exactly what the detector caught (and what it didn't) before you spend tokens running on the full dataset.
4. **The composability story** — generation pipelines create data, production systems create usage traces, and Anonymizer helps make both safer to handle.

In [ ]:
import os

os.environ["DATA_DESIGNER_ASYNC_ENGINE"] = "1"

## Setup

Note the package vs. import naming: dependency is `nemo-anonymizer`, but you import from `anonymizer`. `uv sync` already installed it for you.

In [ ]:
import pandas as pd
from notebook_helpers import (
    ARTIFACT_DIR,
    display_anonymizer_comparison,
    environment_setup,
    show_provider_info,
)

# Pick which hosted backend to use. Leave as "auto" to use whichever key is in
# your .env (precedence: NVIDIA -> OpenRouter -> OpenAI), or set explicitly to
# "nvidia", "openrouter", or "openai" to flip between providers without editing
# any other files.
PROVIDER = "auto"

provider = environment_setup(provider=PROVIDER)
show_provider_info(provider)

## Part 1 — Load production usage data

This notebook is standalone; it does **not** depend on Notebook 2. In a real deployment, these rows would come from observability logs, support transcripts, evaluation traces, or a data warehouse export. For the workshop, we'll use a tiny representative sample inline so everyone starts from the same state.

In [ ]:
usage_df = pd.DataFrame(
    [
        {
            "conversation_id": "prod-2026-05-001",
            "workflow": "account_health",
            "user_message": "Summarize renewal risk for Acme Robotics. The contact is Priya Shah at priya.shah@acme-robotics.example or +1 415-555-0184.",
            "assistant_response": "Acme Robotics shows medium renewal risk. Follow up with Priya Shah about contract MSA-77291 and invoice INV-10482 before June 7.",
            "trace_excerpt": "crm.lookup(account='Acme Robotics', owner='Diego Morales', billing_address='610 Market St, San Francisco, CA 94105')",
        },
        {
            "conversation_id": "prod-2026-05-002",
            "workflow": "support_triage",
            "user_message": "Customer Maria Nguyen from Northstar Analytics says ticket SUP-44819 is blocking payroll export for tenant northstar-prod.",
            "assistant_response": "Escalate SUP-44819 to the integrations queue. Maria Nguyen reported failed exports for Northstar Analytics after SSO changes by Elias Romero.",
            "trace_excerpt": "support.search(email='maria.nguyen@northstar.example', tenant='northstar-prod', ip='203.0.113.42')",
        },
        {
            "conversation_id": "prod-2026-05-003",
            "workflow": "sales_assist",
            "user_message": "Draft a follow-up for Jamal Reed at Redwood BioSystems about quote Q-99317 and the onsite demo at 88 Kendall Square.",
            "assistant_response": "Tell Jamal Reed that Redwood BioSystems can schedule the onsite demo for May 30 and that quote Q-99317 remains valid through June 15.",
            "trace_excerpt": "calendar.create(attendee='jamal.reed@redwood-bio.example', location='88 Kendall Square, Cambridge, MA')",
        },
    ]
)

print(f"\nLoaded {len(usage_df)} production usage rows.")
usage_df[["conversation_id", "workflow", "user_message", "assistant_response", "trace_excerpt"]].head()

🔍 **Look at the `user_message`, `assistant_response`, and `trace_excerpt` columns.** You'll see the kinds of identifiers that show up in production usage data: customer names, employee names, company names, emails, phone numbers, tenant IDs, IP addresses, contract IDs, invoice IDs, and street addresses. Even when the sample is small, the privacy pattern is realistic: operational text often needs scrubbing before it is shared, analyzed, or reused.

That's what Anonymizer is for.

## Part 2 — Configure Anonymizer

Anonymizer is a small, focused tool built on top of Data Designer. Its detection pipeline:

1. **GLiNER-PII** — an NER model that scans the text for entity spans (names, emails, phone numbers, dates, etc.). By default Anonymizer uses the GLiNER-PII variant hosted on NVIDIA Build, so no local model weights -- but you can swap in a self-hosted GLiNER via a `model_providers.yaml` if you need fully offline detection.
2. **LLM augmentation/validation** — a text LLM augments coverage on entities GLiNER misses and validates uncertain spans. Also hosted on NVIDIA Build by default.
3. **Replacement** — applies your chosen strategy to each detected span. `Substitute` uses an LLM call per entity; `Redact` / `Annotate` / `Hash` are local string transforms.

Because both detection stages run on hosted endpoints by default, you only need `NVIDIA_API_KEY` set -- nothing to download.

In [ ]:
from anonymizer import Anonymizer, AnonymizerConfig, AnonymizerInput, Redact, Substitute

anonymizer = Anonymizer()

Anonymizer reads its input from a file path or URL, one text column at a time. We'll write the loaded dataframe to a temp parquet and create an `AnonymizerInput` for each column we want to scrub: `user_message`, `assistant_response`, and `trace_excerpt`.

In [ ]:
tmp_input = ARTIFACT_DIR / "_anonymizer_input.parquet"
tmp_input.parent.mkdir(parents=True, exist_ok=True)
usage_df[["conversation_id", "workflow", "user_message", "assistant_response", "trace_excerpt"]].to_parquet(tmp_input, index=False)

TEXT_COLUMNS = ["user_message", "assistant_response", "trace_excerpt"]
data = {
    col: AnonymizerInput(source=str(tmp_input), text_column=col)
    for col in TEXT_COLUMNS
}
print(f"Anonymizer inputs ready for columns: {TEXT_COLUMNS}")

## Part 3 — Preview with `Redact`

`Redact` replaces every detected entity with a constant template token like `[REDACTED_PERSON]`. It's the simplest strategy: cheap (no LLM call to generate replacements), unambiguous, and obvious to anyone reading the output. The downside is that *training utility* drops — a redacted answer is harder for a model to learn from than a substituted one.

`preview()` runs on a small sample so we can sanity-check before spending tokens on the full dataset.

In [ ]:
redact_config = AnonymizerConfig(replace=Redact())
redact_previews = {
    col: anonymizer.preview(config=redact_config, data=data[col], num_records=3)
    for col in TEXT_COLUMNS
}
for col, preview in redact_previews.items():
    print(f"\n─── {col} ───")
    preview.display_record()

🔍 **What did the detector catch?** The Rich-highlighted display shows each detected entity with its label and replacement. Look critically:

- Did it correctly tag customers, companies, and employees?
- Did it catch operational identifiers like contract IDs, ticket IDs, tenant IDs, emails, and IP addresses?
- Are there false positives (something tagged as PII that isn't actually sensitive)?
- Are there false negatives (something obviously identifying that slipped through)?

PII detection is never perfect. The point of `preview()` is that you see this *before* you commit to the full run.

## Part 4 — Preview with `Substitute`

`Substitute` uses an LLM to generate a *contextually plausible* replacement: "Acme Corp" might become "Globex Industries". This preserves the linguistic shape of the answer — sentence flow, plausibility, downstream training utility — while breaking the link to the original entity.

It costs more (one LLM call per detected entity). Use it when the dataset is going into model training where realism matters.

In [ ]:
substitute_config = AnonymizerConfig(replace=Substitute())
substitute_previews = {
    col: anonymizer.preview(config=substitute_config, data=data[col], num_records=3)
    for col in TEXT_COLUMNS
}
for col, preview in substitute_previews.items():
    print(f"\n─── {col} ───")
    preview.display_record()

## Part 5 — Side by side

Same input, different strategies. Inspect the readability/utility tradeoff.

In [ ]:
for col in TEXT_COLUMNS:
    transformed = f"{col}__transformed"
    originals   = redact_previews[col].dataframe[col].tolist()
    redacted    = redact_previews[col].dataframe[transformed].tolist()
    substituted = substitute_previews[col].dataframe[transformed].tolist()

    print(f"\n{'═' * 20} {col} {'═' * 20}")
    display_anonymizer_comparison(
        original=originals,
        by_strategy={
            "REDACT":     redacted,
            "SUBSTITUTE": substituted,
        },
        max_rows=3,
    )

🔍 **Notice the readability difference.** Redacted usage text is full of `[REDACTED_*]` tokens — clearly anonymised, but stilted. Substituted usage text reads naturally; you'd have to know it was transformed to spot the swap. That naturalness preserves analytical and training signal.

Use `Redact` when the audience is a human reader and clarity matters. Use `Substitute` when the dataset is going into model training and you want preserved language structure.

## Part 6 — Brief mention: `Annotate` and `Hash`

Two more strategies, useful in specific situations:

- **`Annotate`** — keeps the original text but wraps detected entities with a label, e.g. `<Acme Corp-|-organization>`. Useful when you need to preserve content for human review while flagging risk.
- **`Hash`** — replaces each entity with a deterministic hash (e.g. SHA-256, configurable digest length). Same input maps to same output, which preserves *referential integrity* across rows. Useful for analytics where you need to count distinct entities without revealing them.

In [ ]:
from anonymizer import Annotate, Hash

annotate_preview = anonymizer.preview(
    config=AnonymizerConfig(replace=Annotate(format_template="<{text}|{label}>")),
    data=data["user_message"],
    num_records=2,
)
annotate_preview.display_record()

In [ ]:
hash_preview = anonymizer.preview(
    config=AnonymizerConfig(replace=Hash(algorithm="sha256", digest_length=8)),
    data=data["user_message"],
    num_records=2,
)
hash_preview.display_record()

## Part 7 — Full run and save

Pick the strategy that fits your downstream use case. We'll go with `Substitute` here — most realistic for a published training dataset.

In [ ]:
sub_config = AnonymizerConfig(replace=Substitute())
results = {}
for col in TEXT_COLUMNS:
    results[col] = anonymizer.run(config=sub_config, data=data[col])
    print(f"✓ Anonymised {col} ({len(results[col].dataframe)} rows)")

results["assistant_response"].dataframe[["assistant_response", "assistant_response__transformed"]].head(3)

In [ ]:
publishable = usage_df.copy()
for col in TEXT_COLUMNS:
    publishable[f"{col}_anonymized"] = results[col].dataframe[f"{col}__transformed"].tolist()

print(f"{len(publishable)} rows, {len(TEXT_COLUMNS)} columns anonymised.")
publishable[["user_message", "user_message_anonymized", "assistant_response", "assistant_response_anonymized"]].head()

## Recap

In this notebook, you took production-style usage data through a privacy workflow:

**load → detect → preview → compare strategies → anonymize**

The same idea applies to generated datasets, evaluation traces, support logs, sales-assist transcripts, and agent tool traces: inspect a small sample, choose a replacement strategy, then run the full job deliberately.

## Where to go next

- **Production-grade VLM SDG:** the [VLM long-document recipes](https://github.com/NVIDIA-NeMo/DataDesigner/tree/main/docs/assets/recipes/vlm_long_doc) on GitHub are the full version of Notebook 2 — eight stages including OCR, classification-first filtering, multi-page windowed QA, whole-document QA, and a frontier judge. Background story is in the [iterative SDG dev note](https://nvidia-nemo.github.io/DataDesigner/latest/devnotes/training-a-vlm-to-understand-long-documents-an-iterative-sdg-story/).
- **More Data Designer recipes:** [text-to-SQL, code generation, agent search trajectories, deep research](https://github.com/NVIDIA-NeMo/DataDesigner/tree/main/docs/assets/recipes).
- **Anonymizer docs:** [nvidia-nemo.github.io/Anonymizer](https://nvidia-nemo.github.io/Anonymizer/latest/) — including rewrite mode for full-text privacy-preserving rewrites that go beyond per-entity substitution.

## One principle to take with you

Treat dataset creation as engineering. Distributions, dependencies, validation, filtering, privacy — these are pipeline stages, not afterthoughts. The tools matter less than the discipline. *Your training data deserves the same engineering rigour as your model.*

Thanks for coming. 👋